In [2]:
import sys
!{sys.executable} -m pip install pandas openpyxl

  Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached pandas-3.0.2-cp314-cp314-macosx_11_0_arm64.whl (9.9 MB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [openpyxl]━━━━━━━━━ 2/3 [openpyxl]


In [3]:
"""
Convert MITMAPS txt files to Excel format.
Run this in Jupyter Notebook or as a Python script.
Place maps.txt and saos.txt in the same folder as this script.
"""

import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

# ─── CONFIG ──────────────────────────────────────────────────────────────────
MAPS_FILE = "maps.txt"       # change path if needed
SAOS_FILE = "saos.txt"       # change path if needed
MAPS_OUT  = "mitmaps_maps.xlsx"
SAOS_OUT  = "mitmaps_saos.xlsx"

BLUE   = "1F4E79"
LBLUE  = "D6E4F0"
WHITE  = "FFFFFF"
GRAY   = "F2F2F2"

# ─── HELPERS ─────────────────────────────────────────────────────────────────
def apply_header_style(ws, row, n_cols, bg=BLUE, fg=WHITE):
    fill = PatternFill("solid", fgColor=bg)
    font = Font(name="Arial", bold=True, color=fg, size=10)
    for c in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=c)
        cell.fill = fill
        cell.font = font
        cell.alignment = Alignment(horizontal="center", wrap_text=True)

def apply_units_style(ws, row, n_cols):
    fill = PatternFill("solid", fgColor=LBLUE)
    font = Font(name="Arial", italic=True, color=BLUE, size=9)
    for c in range(1, n_cols + 1):
        cell = ws.cell(row=row, column=c)
        cell.fill = fill
        cell.font = font
        cell.alignment = Alignment(horizontal="center")

def parse_step_block(lines):
    """Extract step name, headers, units, and data rows from a step block."""
    step_name = lines[0].strip()
    headers   = [h.strip() for h in lines[1].strip().split("\t")]
    units     = [u.strip() for u in lines[2].strip().split("\t")]
    data_rows = []
    for line in lines[3:]:
        line = line.strip()
        if line and "\t" in line:
            vals = []
            for v in line.split("\t"):
                v = v.strip()
                try:
                    vals.append(float(v))   # converts sci notation to full float
                except ValueError:
                    vals.append(v)
            data_rows.append(vals)
    return step_name, headers, units, data_rows

def write_sheet(ws, title_text, headers, units, data_rows):
    n = len(headers)
    # Title row
    ws.cell(row=1, column=1, value=title_text).font = Font(
        name="Arial", bold=True, size=11, color=BLUE)
    if n > 1:
        ws.merge_cells(f"A1:{get_column_letter(n)}1")
    # Headers
    for c, h in enumerate(headers, 1):
        ws.cell(row=2, column=c, value=h)
    apply_header_style(ws, 2, n)
    # Units
    for c, u in enumerate(units, 1):
        ws.cell(row=3, column=c, value=u)
    apply_units_style(ws, 3, n)
    # Data — write as plain numbers (full decimal, no scientific notation)
    for r, row_vals in enumerate(data_rows, 4):
        for c, val in enumerate(row_vals, 1):
            cell = ws.cell(row=r, column=c, value=val)
            cell.font = Font(name="Arial", size=9)
            if isinstance(val, float):
                # Force full decimal display in Excel
                cell.number_format = "0.000000000000"
    # Column widths
    for c in range(1, n + 1):
        ws.column_dimensions[get_column_letter(c)].width = 22
    ws.freeze_panes = "A4"

# ─── CONVERT SAOS ────────────────────────────────────────────────────────────
print("Converting saos.txt ...")

with open(SAOS_FILE, "r", errors="replace") as f:
    lines = f.readlines()

# Find [step] marker
step_idx = next(i for i, l in enumerate(lines) if "[step]" in l)
block = [l.rstrip("\n\r") for l in lines[step_idx + 1:] if l.strip()]
step_name, headers, units, data_rows = parse_step_block(block)

df_saos = pd.DataFrame(data_rows, columns=headers)

# Write with pandas first, then style with openpyxl
with pd.ExcelWriter(SAOS_OUT, engine="openpyxl") as writer:
    # Write placeholder (we'll overwrite with styled version)
    df_saos.to_excel(writer, sheet_name="SAOS Frequency Sweep", index=False, startrow=2)

wb = load_workbook(SAOS_OUT)
ws = wb["SAOS Frequency Sweep"]

# Clear and rewrite properly
ws.delete_rows(1, ws.max_row)
title = f"MITMAPS SAOS | {step_name} | TA Instruments Discovery HR-3 | 3/6/2020 | {len(data_rows)} frequency points"
write_sheet(ws, title, headers, units, data_rows)
wb.save(SAOS_OUT)
print(f"  Saved: {SAOS_OUT}  ({len(data_rows)} rows, {len(headers)} columns)")

# ─── CONVERT MAPS ────────────────────────────────────────────────────────────
print("\nConverting maps.txt (24 steps x ~26,000 rows) — this may take a few minutes ...")

with open(MAPS_FILE, "r", errors="replace") as f:
    content = f.read()

# Split into steps
raw_steps = re.split(r"\[step\]", content)
raw_steps = [s for s in raw_steps if s.strip()]  # remove empty

print(f"  Found {len(raw_steps)} steps")

# Build summary DataFrame
summary_records = []
all_steps = []

for i, raw in enumerate(raw_steps):
    block = [l.rstrip("\n\r") for l in raw.strip().splitlines() if l.strip()]
    step_name, headers, units, data_rows = parse_step_block(block)
    all_steps.append((step_name, headers, units, data_rows))

    # Sampling rate
    fs_str = dur_str = "N/A"
    if len(data_rows) > 1 and len(data_rows[0]) > 3:
        try:
            t0 = float(data_rows[0][3])
            t1 = float(data_rows[1][3])
            fs_str = str(round(1 / (t1 - t0)))
            dur_str = f"{float(data_rows[-1][3]):.4f}"
        except Exception:
            pass

    summary_records.append({
        "Sheet":         f"Step_{i+1}",
        "Step Name":     step_name,
        "Data Rows":     len(data_rows),
        "Sampling (Hz)": fs_str,
        "Duration (s)":  dur_str,
        "Columns":       len(headers),
    })

df_summary = pd.DataFrame(summary_records)
print(df_summary[["Sheet","Step Name","Data Rows","Sampling (Hz)","Duration (s)"]].to_string(index=False))

# Write Excel
with pd.ExcelWriter(MAPS_OUT, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary", index=False)

wb2 = load_workbook(MAPS_OUT)
ws_sum = wb2["Summary"]

# Style summary sheet
ws_sum.delete_rows(1, ws_sum.max_row)
ws_sum.cell(row=1, column=1, value="MITMAPS — Multi-tone MAPS Rheological Data | github.com/krlennon/mitmaps")
ws_sum["A1"].font = Font(name="Arial", bold=True, size=12, color=BLUE)
ws_sum.merge_cells("A1:F1")

cols = ["Sheet","Step Name","Data Rows","Sampling (Hz)","Duration (s)","Columns"]
for c, col in enumerate(cols, 1):
    ws_sum.cell(row=3, column=c, value=col)
apply_header_style(ws_sum, 3, len(cols))

for i, rec in enumerate(summary_records):
    bg = WHITE if i % 2 == 0 else GRAY
    fill = PatternFill("solid", fgColor=bg)
    for c, key in enumerate(cols, 1):
        cell = ws_sum.cell(row=4+i, column=c, value=rec[key])
        cell.fill = fill
        cell.font = Font(name="Arial", size=10)

for col, w in zip("ABCDEF", [12, 24, 12, 14, 14, 10]):
    ws_sum.column_dimensions[col].width = w

# Write each step as its own sheet
for i, (step_name, headers, units, data_rows) in enumerate(all_steps):
    print(f"  Writing Step {i+1}/{len(all_steps)}: {step_name} ({len(data_rows)} rows)...", end="\r")
    fs = summary_records[i]["Sampling (Hz)"]
    dur = summary_records[i]["Duration (s)"]
    sheet_name = f"Step_{i+1}"
    ws = wb2.create_sheet(sheet_name)
    title = f"{step_name}  |  {fs} Hz  |  {dur} s  |  {len(data_rows)} rows"
    write_sheet(ws, title, headers, units, data_rows)

print(f"\n  Saving {MAPS_OUT} ...")
wb2.save(MAPS_OUT)
print(f"  Saved: {MAPS_OUT}  ({len(all_steps)} sheets + Summary)")

print("\nDone! Both files converted successfully.")
print(f"  {SAOS_OUT}  — 1 sheet, {len(df_saos)} rows")
print(f"  {MAPS_OUT}  — {len(all_steps)} data sheets + Summary")

Converting saos.txt ...
  Saved: mitmaps_saos.xlsx  (16 rows, 13 columns)

Converting maps.txt (24 steps x ~26,000 rows) — this may take a few minutes ...
  Found 25 steps
  Sheet           Step Name  Data Rows Sampling (Hz) Duration (s)
 Step_1   fFilename\tmaps_7         75           N/A          N/A
 Step_2  Arbitrary Wave - 1      26366           488      53.9955
 Step_3  Arbitrary Wave - 2      26366           244     107.9910
 Step_4  Arbitrary Wave - 3      26366           122     215.9820
 Step_5  Arbitrary Wave - 4      26366            61     431.9640
 Step_6  Arbitrary Wave - 5      26366           488      53.9955
 Step_7  Arbitrary Wave - 6      26366           244     107.9910
 Step_8  Arbitrary Wave - 7      26367           122     215.9900
 Step_9  Arbitrary Wave - 8      26366            61     431.9640
Step_10  Arbitrary Wave - 9      26366           488      53.9955
Step_11 Arbitrary Wave - 10      26366           244     107.9910
Step_12 Arbitrary Wave - 11      263